<a href="https://colab.research.google.com/github/RajarapuRamya/first-project/blob/main/Bank_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from abc import ABC, abstractmethod
from enum import Enum
import json
import random
from datetime import datetime


# ==========================
# ENUM
# ==========================
class AccountType(Enum):
    SAVING = "Saving"
    CURRENT = "Current"


# ==========================
# EXCEPTIONS
# ==========================
class InvalidAmountException(Exception):
    pass


class InsufficientBalanceException(Exception):
    pass


class AccountNotFoundException(Exception):
    pass


# ==========================
# TRANSACTION
# ==========================
class Transaction:
    def __init__(self, trans_type, amount):
        self.date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.trans_type = trans_type
        self.amount = amount

    def to_dict(self):
        return {
            "date": self.date,
            "type": self.trans_type,
            "amount": self.amount
        }


# ==========================
# CUSTOMER
# ==========================
class Customer:
    def __init__(self, name, age, mobile):
        self.name = name
        self.age = age
        self.mobile = mobile

    def to_dict(self):
        return {
            "name": self.name,
            "age": self.age,
            "mobile": self.mobile
        }


# ==========================
# ABSTRACT ACCOUNT
# ==========================
class Account(ABC):

    @staticmethod
    def generate_account_number():
        return random.randint(100000, 999999)

    def __init__(self, customer, balance=0):
        self.account_number = self.generate_account_number()
        self.customer = customer
        self._balance = balance
        self.transactions = []

    def deposit(self, amount):
        if amount <= 0:
            raise InvalidAmountException(
                "Deposit amount must be greater than zero."
            )

        self._balance += amount
        self.transactions.append(Transaction("Deposit", amount))

    def withdraw(self, amount):
        if amount <= 0:
            raise InvalidAmountException(
                "Withdrawal amount must be greater than zero."
            )

        if amount > self._balance:
            raise InsufficientBalanceException(
                "Insufficient balance."
            )

        self._balance -= amount
        self.transactions.append(Transaction("Withdraw", amount))

    def get_balance(self):
        return self._balance

    @abstractmethod
    def special_operation(self):
        pass

    def to_dict(self):
        return {
            "account_number": self.account_number,
            "account_type": self.__class__.__name__,
            "customer": self.customer.to_dict(),
            "balance": self._balance,
            "transactions": [
                t.to_dict() for t in self.transactions
            ]
        }


# ==========================
# SAVING ACCOUNT
# ==========================
class SavingAccount(Account):

    interest_rate = 0.05

    def special_operation(self):
        interest = self._balance * self.interest_rate
        self._balance += interest

        self.transactions.append(
            Transaction("Interest", round(interest, 2))
        )


# ==========================
# CURRENT ACCOUNT
# ==========================
class CurrentAccount(Account):

    service_charge = 100

    def special_operation(self):
        self._balance -= self.service_charge

        self.transactions.append(
            Transaction("Service Charge",
                        self.service_charge)
        )


# ==========================
# BANK
# ==========================
class Bank:

    FILE_NAME = "accounts.json"

    def __init__(self):
        self.accounts = []

    def create_account(self,
                       account_type,
                       name,
                       age,
                       mobile,
                       balance):

        customer = Customer(name, age, mobile)

        if account_type == AccountType.SAVING:
            account = SavingAccount(customer, balance)

        else:
            account = CurrentAccount(customer, balance)

        self.accounts.append(account)
        self.save_data()

        print("Account Created Successfully")
        print("Account Number:",
              account.account_number)

    def find_account(self, acc_no):

        for account in self.accounts:
            if account.account_number == acc_no:
                return account

        raise AccountNotFoundException(
            "Account not found."
        )

    def deposit(self, acc_no, amount):
        account = self.find_account(acc_no)
        account.deposit(amount)
        self.save_data()

    def withdraw(self, acc_no, amount):
        account = self.find_account(acc_no)
        account.withdraw(amount)
        self.save_data()

    def transfer(self, from_acc, to_acc, amount):

        sender = self.find_account(from_acc)
        receiver = self.find_account(to_acc)

        sender.withdraw(amount)
        receiver.deposit(amount)

        sender.transactions.append(
            Transaction(
                f"Transfer To {to_acc}",
                amount
            )
        )

        receiver.transactions.append(
            Transaction(
                f"Transfer From {from_acc}",
                amount
            )
        )

        self.save_data()

    def show_accounts(self):

        if not self.accounts:
            print("No Accounts Available")
            return

        for account in self.accounts:

            print("\n---------------------")
            print("Account Number:",
                  account.account_number)
            print("Name:",
                  account.customer.name)
            print("Type:",
                  account.__class__.__name__)
            print("Balance:",
                  account.get_balance())

    def mini_statement(self, acc_no):

        account = self.find_account(acc_no)

        print("\nTransaction History")

        for t in account.transactions[-5:]:

            print(
                t.date,
                t.trans_type,
                t.amount
            )

    def reports(self):

        if not self.accounts:
            print("No Accounts Found")
            return

        total_balance = sum(
            acc.get_balance()
            for acc in self.accounts
        )

        saving_count = sum(
            isinstance(acc, SavingAccount)
            for acc in self.accounts
        )

        current_count = sum(
            isinstance(acc, CurrentAccount)
            for acc in self.accounts
        )

        highest = max(
            self.accounts,
            key=lambda x: x.get_balance()
        )

        lowest = min(
            self.accounts,
            key=lambda x: x.get_balance()
        )

        print("\n===== REPORT =====")
        print("Total Accounts:",
              len(self.accounts))
        print("Saving Accounts:",
              saving_count)
        print("Current Accounts:",
              current_count)
        print("Total Balance:",
              total_balance)

        print("\nHighest Balance")
        print(highest.customer.name,
              highest.get_balance())

        print("\nLowest Balance")
        print(lowest.customer.name,
              lowest.get_balance())

    def save_data(self):

        data = [
            acc.to_dict()
            for acc in self.accounts
        ]

        with open(
            self.FILE_NAME,
            "w"
        ) as file:
            json.dump(
                data,
                file,
                indent=4
            )


# ==========================
# MAIN MENU
# ==========================
def main():

    bank = Bank()

    while True:

        print("\n===== BANK MANAGEMENT =====")
        print("1. Create Saving Account")
        print("2. Create Current Account")
        print("3. View Accounts")
        print("4. Deposit")
        print("5. Withdraw")
        print("6. Transfer")
        print("7. Mini Statement")
        print("8. Reports")
        print("0. Exit")

        choice = input("Enter Choice: ")

        try:

            if choice == "1":

                name = input("Name: ")
                age = int(input("Age: "))
                mobile = input("Mobile: ")
                balance = float(
                    input("Initial Balance: ")
                )

                bank.create_account(
                    AccountType.SAVING,
                    name,
                    age,
                    mobile,
                    balance
                )

            elif choice == "2":

                name = input("Name: ")
                age = int(input("Age: "))
                mobile = input("Mobile: ")
                balance = float(
                    input("Initial Balance: ")
                )

                bank.create_account(
                    AccountType.CURRENT,
                    name,
                    age,
                    mobile,
                    balance
                )

            elif choice == "3":
                bank.show_accounts()

            elif choice == "4":

                acc = int(
                    input("Account Number: ")
                )

                amt = float(
                    input("Amount: ")
                )

                bank.deposit(acc, amt)

            elif choice == "5":

                acc = int(
                    input("Account Number: ")
                )

                amt = float(
                    input("Amount: ")
                )

                bank.withdraw(acc, amt)

            elif choice == "6":

                from_acc = int(
                    input("From Account: ")
                )

                to_acc = int(
                    input("To Account: ")
                )

                amt = float(
                    input("Amount: ")
                )

                bank.transfer(
                    from_acc,
                    to_acc,
                    amt
                )

            elif choice == "7":

                acc = int(
                    input("Account Number: ")
                )

                bank.mini_statement(acc)

            elif choice == "8":
                bank.reports()

            elif choice == "0":
                print("Thank You")
                break

            else:
                print("Invalid Choice")

        except Exception as e:
            print("Error:", e)


if __name__ == "__main__":
    main()


===== BANK MANAGEMENT =====
1. Create Saving Account
2. Create Current Account
3. View Accounts
4. Deposit
5. Withdraw
6. Transfer
7. Mini Statement
8. Reports
0. Exit
Enter Choice: 1
Name: ramya
Age: 21
Mobile: 1234567890
Initial Balance: 50000
Account Created Successfully
Account Number: 392875

===== BANK MANAGEMENT =====
1. Create Saving Account
2. Create Current Account
3. View Accounts
4. Deposit
5. Withdraw
6. Transfer
7. Mini Statement
8. Reports
0. Exit
Enter Choice: 2
Name: ramya
Age: 21
Mobile: 1234567890
Initial Balance: 25000
Account Created Successfully
Account Number: 919390

===== BANK MANAGEMENT =====
1. Create Saving Account
2. Create Current Account
3. View Accounts
4. Deposit
5. Withdraw
6. Transfer
7. Mini Statement
8. Reports
0. Exit
Enter Choice: 3

---------------------
Account Number: 392875
Name: ramya
Type: SavingAccount
Balance: 50000.0

---------------------
Account Number: 919390
Name: ramya
Type: CurrentAccount
Balance: 25000.0

===== BANK MANAGEMENT ===